# Load Tree Equity Data into Database

Run this notebook to load the processed Tree Equity Score data into your DuckDB database.

In [5]:
import duckdb
import pandas as pd
from pathlib import Path

# Close any existing connection first
try:
    conn.close()
except:
    pass

# Configuration
DB_PATH = Path('../data/db/nj_pipeline.duckdb')
PROCESSED_FILE = Path('../data/processed/tree_equity/tree_equity_nj_processed.csv')

print("Loading processed Tree Equity data...")

Loading processed Tree Equity data...


In [6]:
# Load the processed data
df = pd.read_csv(PROCESSED_FILE)
print(f"Loaded {len(df):,} block groups")
print(f"\nColumns: {list(df.columns)}")
display(df.head())

,block_group_id,place,state,state_abbr,county,ua_name,ua_pop,congressio,cbg_pop,acs_pop,...,rankgrpsz,_bld1200,_veg1200,_tot1200,_bld1500,_veg1500,_tot1500,_bld1800,_veg1800,_tot1800
0,340010001001,Atlantic City,New Jersey,NJ,Atlantic County,"Atlantic City--Ocean City--Villas, NJ",294900,NJ Congressional District 2,1094,811.0,...,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,340010001002,Atlantic City,New Jersey,NJ,Atlantic County,"Atlantic City--Ocean City--Villas, NJ",294900,NJ Congressional District 2,1240,1303.0,...,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,340010002001,Atlantic City,New Jersey,NJ,Atlantic County,"Atlantic City--Ocean City--Villas, NJ",294900,NJ Congressional District 2,1171,1901.0,...,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,340010002002,Atlantic City,New Jersey,NJ,Atlantic County,"Atlantic City--Ocean City--Villas, NJ",294900,NJ Congressional District 2,986,560.0,...,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,340010002003,Atlantic City,New Jersey,NJ,Atlantic County,"Atlantic City--Ocean City--Villas, NJ",294900,NJ Congressional District 2,740,953.0,...,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Connect to database
conn = duckdb.connect(str(DB_PATH))
print("Connected to database")

IOException: IO Error: Could not set lock on file "/Users/caiotormin/torm/nj_pipeline/notebooks/../data/db/nj_pipeline.duckdb": Conflicting lock is held in /opt/anaconda3/envs/nj-pipeline/bin/python3.11 (PID 52464) by user caiotormin. However, you would be able to open this database in read-only mode, e.g. by using the -readonly parameter in the CLI. See also https://duckdb.org/docs/stable/connect/concurrency

In [ ]:
# Create block group level table
conn.execute("DROP TABLE IF EXISTS tree_equity_blockgroups")
conn.execute("CREATE TABLE tree_equity_blockgroups AS SELECT * FROM df")

count = conn.execute("SELECT COUNT(*) FROM tree_equity_blockgroups").fetchone()[0]
print(f"✓ Created tree_equity_blockgroups with {count:,} records")

In [ ]:
# Show schema
schema = conn.execute("DESCRIBE tree_equity_blockgroups").df()
display(schema)

In [ ]:
# Create aggregation by place (city/town)
print("Creating aggregation by place...")

conn.execute("""
CREATE OR REPLACE TABLE tree_equity_by_place AS
SELECT
    place,
    county,
    COUNT(*) as num_block_groups,
    AVG(treecanopy) as avg_tree_canopy_pct,
    MIN(treecanopy) as min_tree_canopy_pct,
    MAX(treecanopy) as max_tree_canopy_pct,
    AVG(tes) as avg_tree_equity_score,
    MIN(tes) as min_tree_equity_score,
    MAX(tes) as max_tree_equity_score,
    AVG(priority_i) as avg_priority_index,
    AVG(pctpoc) as avg_pct_poc,
    AVG(pctpov) as avg_pct_poverty,
    AVG(temp_diff) as avg_temp_difference
FROM tree_equity_blockgroups
WHERE place IS NOT NULL
GROUP BY place, county
ORDER BY place
""")

place_count = conn.execute("SELECT COUNT(*) FROM tree_equity_by_place").fetchone()[0]
print(f"✓ Created tree_equity_by_place with {place_count} places")

In [ ]:
# Create aggregation by county
print("Creating aggregation by county...")

conn.execute("""
CREATE OR REPLACE TABLE tree_equity_by_county AS
SELECT
    county,
    COUNT(*) as num_block_groups,
    AVG(treecanopy) as avg_tree_canopy_pct,
    MIN(treecanopy) as min_tree_canopy_pct,
    MAX(treecanopy) as max_tree_canopy_pct,
    AVG(tes) as avg_tree_equity_score,
    MIN(tes) as min_tree_equity_score,
    MAX(tes) as max_tree_equity_score,
    AVG(priority_i) as avg_priority_index,
    AVG(pctpoc) as avg_pct_poc,
    AVG(pctpov) as avg_pct_poverty,
    AVG(temp_diff) as avg_temp_difference
FROM tree_equity_blockgroups
GROUP BY county
ORDER BY county
""")

county_count = conn.execute("SELECT COUNT(*) FROM tree_equity_by_county").fetchone()[0]
print(f"✓ Created tree_equity_by_county with {county_count} counties")

In [ ]:
# View top places by tree equity score
print("\nTop 10 places by tree equity score:")
top_places = conn.execute("""
    SELECT place, county, num_block_groups, 
           ROUND(avg_tree_canopy_pct, 1) as tree_canopy_pct,
           ROUND(avg_tree_equity_score, 1) as equity_score
    FROM tree_equity_by_place
    ORDER BY avg_tree_equity_score DESC
    LIMIT 10
""").df()
display(top_places)

In [ ]:
# View county-level data
print("\nTree equity by county:")
counties = conn.execute("""
    SELECT county, num_block_groups,
           ROUND(avg_tree_canopy_pct, 1) as tree_canopy_pct,
           ROUND(avg_tree_equity_score, 1) as equity_score
    FROM tree_equity_by_county
    ORDER BY avg_tree_equity_score DESC
""").df()
display(counties)

In [ ]:
# Verify all tree tables
print("\nAll tree-related tables:")
tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    AND table_name LIKE '%tree%'
    ORDER BY table_name
""").df()
display(tables)

## Important Note

The Tree Equity data is at the **Census block group** level, not ZCTA level.

**Available tables:**
- `tree_equity_blockgroups` - Full block group detail (6,363 records)
- `tree_equity_by_place` - Aggregated by city/town
- `tree_equity_by_county` - Aggregated by county

**To join with ZCTA-level data (CDC PLACES, Zillow):**
- Use county-level joins
- Or manually map major cities to their primary ZCTAs

Example county-level join:
```sql
SELECT
    z.zip_code,
    z.county,
    z.zhvi_latest,
    c.obesity,
    t.avg_tree_canopy_pct,
    t.avg_tree_equity_score
FROM zillow_zipcode_latest z
LEFT JOIN cdc_places_wide c ON z.zip_code = c.zcta AND c.year = 2023
LEFT JOIN tree_equity_by_county t ON z.county = t.county
WHERE z.zhvi_latest IS NOT NULL
```

In [ ]:
conn.close()
print("✓ Tree Equity data successfully loaded!")
print("\nNow you can use the eda_tree_equity.ipynb notebook for analysis.")